# Rank, Nullity, Subspaces, Determinant and Chemical Reactions
### A Linear Algebra Lab for Math 107

---

Welcome! In this lab you'll explore three of the most important ideas in linear algebra: **rank**, **nullity**, and **subspaces** as well as consider surprising application: balancing chemical equations.

It turns out that finding the right coefficients to balance a chemical reaction is *exactly* the same problem as finding the null space of a matrix. By the end of this lab, you'll see why.

Every step is explained, and you'll mostly just be **modifying existing code** rather than writing it from scratch.

The notebook has a few custom-written functions. You are **not** responsible for understanding that code. Just run the cell and you'll be able to use the function in later cells.

**How to use this notebook:**
- Each gray box (called a **cell**) contains either explanation text or Python code.
- To **run a code cell**, click on it and press **Shift + Enter** (or click the ▶ button).
- Run cells **in order from top to bottom** — later cells depend on earlier ones.
- Don't be afraid to experiment! You can always re-run a cell after changing it.

**Estimated time:** around 60–75 minutes

The lab covers the following topics in order:

- Rank and what it measures
- The null space and nullity
- The Rank-Nullity Theorem — discovering it experimentally
- The column space and when $A\mathbf{x} = \mathbf{b}$ has a solution
- The determinantand the area of a parallelogram
- Application: balancing chemical equations with the null space
- Practice problems

---

## Step 0: Setting Up: Run This First!

In [ ]:
# Lines starting with # are comments — Python ignores them.

import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

print("Libraries loaded successfully! You're ready to go.")

---

## Part 1: Rank: What Does It Really Tell Us?

### 1.1 Review: What Is Rank?

The **rank** of a matrix $A$ is the number of **pivot columns** in its row echelon form.  Equivalently, it is the number of linearly independent columns in the matrix.

You can think of rank as measuring how much independent information is packed into the matrix. A matrix with lots of redundant columns has low rank.

In NumPy, `np.linalg.matrix_rank(A)` computes this for us instantly.

Let's experiment with a few examples.

In [ ]:
# Example 1: A full-rank matrix
A1 = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
], dtype=float)

print("Matrix A1 (identity matrix):")
print(A1)
print(f"Rank of A1: {np.linalg.matrix_rank(A1)}")
print(f"Size of A1: {A1.shape[0]} rows, {A1.shape[1]} columns")
print()

In [ ]:
# Example 2: A matrix where one row is a multiple of another
A2 = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [2, 4, 6]    # <-- this is exactly 2 times row 1!
], dtype=float)

print("Matrix A2 (row 3 = 2 * row 1):")
print(A2)
print(f"Rank of A2: {np.linalg.matrix_rank(A2)}")
print()
print("Notice: rank dropped by 1 because row 3 is redundant.")

In [ ]:
# Example 3: A matrix where ALL rows are multiples of the first
A3 = np.array([
    [1, 2, 3],
    [2, 4, 6],
    [3, 6, 9]
], dtype=float)

print("Matrix A3 (all rows are multiples of [1, 2, 3]):")
print(A3)
print(f"Rank of A3: {np.linalg.matrix_rank(A3)}")
print()
print("Only one truly independent row — rank is 1.")

### 1.2 Rank and Non-Square Matrices

Rank applies to **any** matrix, not just square matrices. For an $m \times n$ matrix:
- Rank can be **at most** $\min(m, n)$ because you can't have more independent directions than the smaller dimension allows.
- When rank equals $\min(m, n)$, we say the matrix has **full rank**.

In [ ]:
# A tall matrix (more rows than columns): 5 x 3
A4 = np.array([
    [1, 2, 0],
    [0, 1, 1],
    [1, 0, 3],
    [2, 1, 4],
    [0, 3, 1]
], dtype=float)

m, n = A4.shape
r = np.linalg.matrix_rank(A4)
print(f"Matrix A4: {m} rows x {n} columns")
print(f"Rank: {r}")
print(f"Maximum possible rank: min({m}, {n}) = {min(m, n)}")
print(f"Is A4 full rank? {r == min(m, n)}")
print()

# A wide matrix (more columns than rows): 3 x 5
A5 = np.array([
    [1, 0, 2, 1, 3],
    [0, 1, 1, 2, 0],
    [1, 1, 3, 3, 3]
], dtype=float)

m2, n2 = A5.shape
r2 = np.linalg.matrix_rank(A5)
print(f"Matrix A5: {m2} rows x {n2} columns")
print(f"Rank: {r2}")
print(f"Maximum possible rank: min({m2}, {n2}) = {min(m2, n2)}")
print(f"Is A5 full rank? {r2 == min(m2, n2)}")

---

## Part 2: The Null Space and Nullity

### 2.1 What Is the Null Space?

The **null space** of a matrix $A$ (also written $\text{Nul}(A)$) is the set of all vectors $\mathbf{x}$ such that:

$$A\mathbf{x} = \mathbf{0}$$

In other words: which inputs does $A$ "crush to zero"?

The null space is always a **subspace** (it contains $\mathbf{0}$, it's closed under addition, and closed under scalar multiplication).

The **nullity** is the dimension of the null space (how many linearly independent vectors span it).

### 2.2 Computing a Basis for the Null Space

SciPy can compute a basis for the null space using `linalg.null_space(A)`. The columns of the result are basis vectors for $\text{Nul}(A)$.

The function below wraps this and displays the results cleanly. Run the cell to define it.

In [ ]:
# ---------------------------------------------------------------
# This function computes and displays the null space of a matrix.
# You do NOT need to modify or understand this — just run it.
# ---------------------------------------------------------------
def show_null_space(A, name="A", tol=1e-10):
    """
    Computes a basis for the null space of A.
    Prints the basis vectors and verifies A @ v ≈ 0 for each.
    Returns the null space basis matrix.
    """
    ns = linalg.null_space(A)
    # Zero out tiny numerical noise
    ns[np.abs(ns) < tol] = 0.0
    nullity = ns.shape[1]
    rank = np.linalg.matrix_rank(A)
    m, n = A.shape

    print(f"Matrix {name}: {m} rows x {n} columns")
    print(f"  Rank    = {rank}")
    print(f"  Nullity = {nullity}  (dimension of the null space)")
    print()

    if nullity == 0:
        print(f"  Null space = {{0}}  (only the zero vector)")
    else:
        print(f"  Null space basis ({nullity} vector(s)):")
        for i in range(nullity):
            v = ns[:, i]
            residual = np.max(np.abs(A @ v))
            print(f"    v{i+1} = {v}")
            print(f"         A @ v{i+1} = {A @ v}   (max error: {residual:.2e})")
    print()
    return ns

print("Function 'show_null_space' defined and ready!")

In [ ]:
# Null space of the full-rank identity matrix
# If A is invertible, Ax=0 has ONLY the trivial solution x=0
print("=== Full rank matrix: A1 (identity) ===")
show_null_space(A1, name="A1")

In [ ]:
# Null space of the rank-deficient matrix A2
print("=== Rank-deficient matrix: A2 ===")
ns2 = show_null_space(A2, name="A2")

In [ ]:
# Null space of the wide matrix A5 (more columns than rows)
# Wide matrices almost always have a non-trivial null space!
print("=== Wide matrix: A5 (3 x 5) ===")
ns5 = show_null_space(A5, name="A5")

### 2.3 What Does the Null Space Look Like Geometrically?

For a matrix $A$ acting on $\mathbb{R}^2$, we can actually draw the null space.

Run the cell below. It tries several $2 \times 2$ matrices and plots the null space (the set of all $\mathbf{x}$ with $A\mathbf{x} = \mathbf{0}$) as a point, a line, or the plane).

In [ ]:
# ---------------------------------------------------------------
# Visualization: null space in R^2
# You do NOT need to modify this — just run it.
# ---------------------------------------------------------------
def plot_null_space_2d(matrices, titles):
    fig, axes = plt.subplots(1, len(matrices), figsize=(4*len(matrices), 4))
    if len(matrices) == 1:
        axes = [axes]
    t = np.linspace(-3, 3, 300)

    for ax, A, title in zip(axes, matrices, titles):
        ns = linalg.null_space(A)
        rank = np.linalg.matrix_rank(A)
        nullity = ns.shape[1]

        ax.axhline(0, color='lightgray', linewidth=1)
        ax.axvline(0, color='lightgray', linewidth=1)
        ax.set_xlim(-3, 3)
        ax.set_ylim(-3, 3)
        ax.set_aspect('equal')
        ax.set_title(f"{title}\nrank={rank}, nullity={nullity}", fontsize=11)
        ax.set_xlabel("x₁")
        ax.set_ylabel("x₂")

        if nullity == 0:
            ax.plot(0, 0, 'ro', markersize=12, label='Null space = {0}')
            ax.legend(fontsize=9)
        elif nullity == 1:
            v = ns[:, 0]
            ax.plot(t*v[0], t*v[1], 'r-', linewidth=2.5,
                    label=f'Null space\n(line through 0)')
            ax.plot(0, 0, 'ko', markersize=6)
            ax.legend(fontsize=9)
        else:  # nullity == 2 — all of R^2
            ax.fill_between([-3,3],[-3,-3],[3,3], alpha=0.2, color='red',
                            label='Null space = all of ℝ²')
            ax.legend(fontsize=9)

    plt.suptitle("Null Space of 2×2 Matrices", fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

mats = [
    np.array([[1, 0],[0, 1]], dtype=float),   # identity: rank 2
    np.array([[1, 2],[2, 4]], dtype=float),   # rank 1
    np.array([[0, 0],[0, 0]], dtype=float),   # zero matrix: rank 0
]
titles = [
    "Identity\n[[1,0],[0,1]]",
    "Rank-1\n[[1,2],[2,4]]",
    "Zero matrix\n[[0,0],[0,0]]",
]
plot_null_space_2d(mats, titles)

**Key geometric takeaway:**
- **Nullity 0** → null space is just the origin (a single point). The matrix is invertible.
- **Nullity 1** → null space is a **line** through the origin.
- **Nullity 2** → null space is all of $\mathbb{R}^2$ (the matrix sends everything to zero).

Notice that the null space is always a **subspace**: it always contains the origin, and any line through the origin in $\mathbb{R}^2$ is a subspace.  The other cases are the "extreme" versions of a subspace: just the zero vector or the whole space.

---

## Part 3: The Rank-Nullity Theorem

### 3.1 Discovering the Pattern

So far we've seen rank and nullity separately. Let's look at them together and see if we can spot a pattern.

For each matrix below, we'll compute the rank, the nullity, and the number of columns $n$.

In [ ]:
# A collection of matrices of different shapes and ranks
test_matrices = [
    np.array([[1,0,0],[0,1,0],[0,0,1]], dtype=float),          # 3x3 identity
    np.array([[1,2,3],[4,5,6],[2,4,6]], dtype=float),          # 3x3 rank 2
    np.array([[1,2,3],[2,4,6],[3,6,9]], dtype=float),          # 3x3 rank 1
    np.array([[1,0,2,1],[0,1,1,2],[1,1,3,3]], dtype=float),    # 3x4
    np.array([[1,2,0,1,3],[0,1,1,2,0],[1,0,3,1,2],
              [2,1,0,3,1]], dtype=float),                      # 4x5
    np.array([[1,3],[2,6],[4,12]], dtype=float),               # 3x2 rank 1
    np.array([[1,0],[0,1],[1,1]], dtype=float),                # 3x2 rank 2
]

print(f"{'Shape':<12} {'n (cols)':<12} {'Rank':<10} {'Nullity':<10} {'Rank + Nullity':<15}")
print("-" * 60)

for A in test_matrices:
    m, n = A.shape
    r = np.linalg.matrix_rank(A)
    ns = linalg.null_space(A)
    nullity = ns.shape[1]
    print(f"{str(A.shape):<12} {n:<12} {r:<10} {nullity:<10} {r + nullity:<15}")

### 3.2 The Theorem

Do you see the pattern? The last column is always equal to $n$, the number of columns!

This is the **Rank-Nullity Theorem**, one of the most fundamental results in linear algebra:

$$\boxed{\text{rank}(A) + \text{nullity}(A) = n}$$

where $n$ is the number of **columns** of $A$.

**Intuition:** The columns of $A$ define $n$ directions in the input space $\mathbb{R}^n$. Some of those directions get "used" (they map to independent directions in the output and those contribute to the rank). The rest get "crushed to zero" (they form the null space and contribute to the nullity). Together they account for all $n$ directions.

### 3.3 Using the Theorem to Predict

The rank-nullity theorem lets us figure out the nullity without computing the null space.  We can find it just from the rank.

In [ ]:
# Example: a 4 x 7 matrix with rank 3.
# What must the nullity be?

B = np.array([
    [1, 2, 0, 1, 3, 0, 1],
    [0, 0, 1, 2, 1, 1, 0],
    [1, 2, 1, 3, 4, 1, 1],    # row 3 = row 1 + row 2
    [0, 0, 2, 4, 2, 2, 0],    # row 4 = 2 * row 2
], dtype=float)

m, n = B.shape
r = np.linalg.matrix_rank(B)
predicted_nullity = n - r

print(f"Matrix B: {m} rows x {n} columns")
print(f"Rank = {r}")
print(f"Predicted nullity = n - rank = {n} - {r} = {predicted_nullity}")
print()

# Verify by actually computing the null space
ns_B = linalg.null_space(B)
actual_nullity = ns_B.shape[1]
print(f"Actual nullity (from null_space) = {actual_nullity}")
print(f"Prediction correct? {predicted_nullity == actual_nullity}")

---

## Part 4: The Column Space: When Does $A\mathbf{x} = \mathbf{b}$ Have a Solution?

### 4.1 The Column Space

The **column space** of $A$ (written $\text{Col}(A)$) is the set of all vectors of the form $A\mathbf{x}$ It is all possible **outputs** of the matrix.

It's called the column space because it's exactly the span of the columns of $A$.

The dimension of the column space is... the **rank**! (That's one reason rank is so important.)

**Key fact:** The equation $A\mathbf{x} = \mathbf{b}$ has a solution **if and only if $\mathbf{b}$ is in the column space of $A$**.

### 4.2 Testing Whether $\mathbf{b}$ Is in the Column Space

In [ ]:
# ---------------------------------------------------------------
# This function tests whether Ax = b has a solution, by checking
# if b is in the column space of A.
# You do NOT need to modify or understand this — just run it.
# ---------------------------------------------------------------
def is_in_column_space(A, b, name_b="b"):
    """
    Checks whether b is in the column space of A (i.e., whether Ax = b is consistent).
    Uses the fact that [A | b] and A have the same rank iff b is in Col(A).
    """
    rank_A = np.linalg.matrix_rank(A)
    Ab = np.column_stack([A, b])
    rank_Ab = np.linalg.matrix_rank(Ab)

    in_col_space = (rank_A == rank_Ab)

    print(f"  rank(A)       = {rank_A}")
    print(f"  rank([A | {name_b}]) = {rank_Ab}")
    if in_col_space:
        print(f"  → {name_b} IS in Col(A).  Ax = {name_b} HAS a solution.")
    else:
        print(f"  → {name_b} is NOT in Col(A).  Ax = {name_b} has NO solution.")
    return in_col_space

print("Function 'is_in_column_space' defined and ready!")

In [ ]:
# Let's test with a specific matrix
A = np.array([
    [1, 2, 1],
    [2, 1, 3],
    [3, 3, 4]
], dtype=float)

# Note: row 3 = row 1 + row 2, so rank is 2. Col(A) is a 2D plane in R^3.
print(f"rank(A) = {np.linalg.matrix_rank(A)}")
print()

# b1 = column 1 of A — definitely in Col(A)
b1 = np.array([1, 2, 3], dtype=float)
print("Testing b1 = [1, 2, 3] (first column of A):")
is_in_column_space(A, b1, "b1")
print()

# b2 = a vector NOT in Col(A)
b2 = np.array([1, 0, 0], dtype=float)
print("Testing b2 = [1, 0, 0]:")
is_in_column_space(A, b2, "b2")

---

## Part 5: Application — Balancing Chemical Equations

### 5.1 The Chemistry Problem

In chemistry, a **balanced equation** has the same number of each type of atom on both sides of the reaction.

For example, burning methane (CH₄) in oxygen (O₂):

$$x_1 \, \text{CH}_4 + x_2 \, \text{O}_2 \longrightarrow x_3 \, \text{CO}_2 + x_4 \, \text{H}_2\text{O}$$

We need to find positive integers $x_1, x_2, x_3, x_4$ such that atom counts balance:

| Atom | Left side | Right side |
|------|-----------|------------|
| **C** | $x_1$ | $x_3$ |
| **H** | $4x_1$ | $2x_4$ |
| **O** | $2x_2$ | $2x_3 + x_4$ |

Setting left = right gives us a **homogeneous linear system** (everything moved to one side, equals zero):

$$\begin{aligned}
x_1 - x_3 &= 0 & \text{(Carbon)}\\
4x_1 - 2x_4 &= 0 & \text{(Hydrogen)}\\
2x_2 - 2x_3 - x_4 &= 0 & \text{(Oxygen)}
\end{aligned}$$

In matrix form: $A\mathbf{x} = \mathbf{0}$. **The solution is in the null space of $A$!**

### 5.2 Building the Atom Matrix

Let's set up the matrix. Each **row** is one type of atom. Each **column** is one molecule. Reactants get **positive** coefficients, products get **negative** (because they move to the other side when we set everything equal to zero).

In [ ]:
# Reaction: CH4 + O2 → CO2 + H2O
# Columns: CH4,  O2,  CO2, H2O
# Rows: C, H, O

#               CH4   O2   CO2   H2O
A_methane = np.array([
    [ 1,    0,   -1,    0  ],   # Carbon:   1 in CH4, -1 in CO2
    [ 4,    0,    0,   -2  ],   # Hydrogen: 4 in CH4, -2 in H2O
    [ 0,    2,   -2,   -1  ],   # Oxygen:   2 in O2, -2 in CO2, -1 in H2O
], dtype=float)

print("Atom matrix for CH4 + O2 → CO2 + H2O:")
print("         CH4   O2   CO2   H2O")
print("Carbon  ", A_methane[0])
print("Hydrogen", A_methane[1])
print("Oxygen  ", A_methane[2])
print()
print(f"Shape: {A_methane.shape}")
print(f"Rank: {np.linalg.matrix_rank(A_methane)}")
print(f"Nullity (predicted): {A_methane.shape[1] - np.linalg.matrix_rank(A_methane)}")

The nullity is 1.  There's exactly one free variable, which means there's a **one-dimensional family** of solutions. This makes sense: if $(1, 2, 1, 2)$ balances the equation, so does $(2, 4, 2, 4)$. The only unique balanced equation is the one with **smallest positive integer coefficients**.

In [ ]:
# ---------------------------------------------------------------
# This function finds the null space, then scales the solution
# to get the smallest positive integer coefficients.
# You do NOT need to modify or understand this — just run it.
# ---------------------------------------------------------------
def balance_equation(atom_matrix, molecule_names, reaction_str):
    """
    Given an atom matrix (rows=atoms, cols=molecules, reactants positive,
    products negative), finds the balanced integer coefficients.
    """
    print(f"Balancing: {reaction_str}")
    print()

    ns = linalg.null_space(atom_matrix)
    nullity = ns.shape[1]

    if nullity == 0:
        print("  Null space is trivial — this reaction cannot be balanced!")
        return None
    if nullity > 1:
        print(f"  Nullity = {nullity}: multiple independent solutions exist.")
        print("  This reaction may involve multiple independent sub-reactions.")

    # Take the first null space vector
    v = ns[:, 0]

    # Make all entries positive (flip sign if needed)
    if v[0] < 0:
        v = -v

    # Scale to integers: divide by smallest, then round
    v = v / np.min(np.abs(v[np.abs(v) > 1e-10]))
    # Find a scale factor to clear fractions (up to denominator 20)
    for denom in range(1, 21):
        scaled = v * denom
        if np.allclose(scaled, np.round(scaled), atol=1e-6):
            v = np.round(scaled).astype(int)
            break

    print(f"  Null space vector (raw):  {ns[:, 0]}")
    print(f"  Scaled to integers:       {v}")
    print()
    print("  Balanced equation:")
    terms = [f"{v[i]} {molecule_names[i]}" for i in range(len(molecule_names))]
    # Find where products start (negative in original matrix)
    # We just print all terms with their coefficients
    print(f"    {reaction_str}")
    coeff_str = "  →  Coefficients: " + ",  ".join(terms)
    print(coeff_str)
    print()

    # Verify: A @ v should be zero
    residual = np.max(np.abs(atom_matrix @ v))
    print(f"  Verification: max|A·x| = {residual:.2e}  ", end="")
    print("✓ Balanced!" if residual < 1e-6 else "✗ Error — something went wrong.")
    return v

print("Function 'balance_equation' defined and ready!")

In [ ]:
# Balance the methane combustion reaction!
molecules_methane = ['CH4', 'O2', 'CO2', 'H2O']
coeffs = balance_equation(
    A_methane,
    molecules_methane,
    "CH4 + O2 → CO2 + H2O"
)

print()
print(f"The balanced equation is:")
print(f"  {coeffs[0]} CH4  +  {coeffs[1]} O2  →  {coeffs[2]} CO2  +  {coeffs[3]} H2O")

The balanced equation $\text{CH}_4 + 2\text{O}_2 \to \text{CO}_2 + 2\text{H}_2\text{O}$ is correct!

The null space vector directly gave us the coefficients. This is what makes the null space powerful. It's not just an abstract set of vectors, it's the **solution set** of a real problem.

### 5.3 A More Complex Reaction

Let's try a more complicated reaction: the combustion of **ethanol** (C₂H₅OH):

$$x_1 \, \text{C}_2\text{H}_5\text{OH} + x_2 \, \text{O}_2 \longrightarrow x_3 \, \text{CO}_2 + x_4 \, \text{H}_2\text{O}$$

Ethanol has 2 carbons, 6 hydrogens, and 1 oxygen.

In [ ]:
# Ethanol combustion: C2H5OH + O2 → CO2 + H2O
# C2H5OH has: C=2, H=6, O=1
#
#                C2H5OH  O2   CO2   H2O
A_ethanol = np.array([
    [  2,    0,   -1,    0  ],   # Carbon
    [  6,    0,    0,   -2  ],   # Hydrogen
    [  1,    2,   -2,   -1  ],   # Oxygen
], dtype=float)

balance_equation(
    A_ethanol,
    ['C2H5OH', 'O2', 'CO2', 'H2O'],
    "C2H5OH + O2 → CO2 + H2O"
)

# Part 6: The Determinant and the Area of a Parallelogram

Here's a geometric way to think about linear dependence.

Take any two 2D vectors (the columns of a 2×2 matrix). Together they span a **parallelogram**. The absolute value of the **determinant** of that matrix equals the **area** of that parallelogram.

This gives us a vivid way to see what rank < 2 really means: if the two columns point in the same (or opposite) direction, the parallelogram **collapses** to a flat line — zero area, zero determinant, rank 1.

$$\det\begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad - bc$$

Run the cell below and try changing the vectors to see this happen live.

In [ ]:
def plot_parallelogram(v1, v2):
    """
    Plots two vectors v1, v2 as arrows from the origin,
    shades the parallelogram they span, and labels its area = |det|.
    """
    A = np.column_stack([v1, v2])
    det = np.linalg.det(A)
    area = abs(det)

    # corners of the parallelogram: 0, v1, v1+v2, v2
    corners = np.array([
        [0, 0],
        v1,
        v1 + v2,
        v2,
        [0, 0]
    ])

    fig, ax = plt.subplots(figsize=(5, 5))

    # shaded parallelogram
    ax.fill(corners[:, 0], corners[:, 1],
            alpha=0.25, color='steelblue', label='parallelogram')
    ax.plot(corners[:, 0], corners[:, 1], 'steelblue', linewidth=1)

    # vectors as arrows
    arrow_kw = dict(head_width=0.12, head_length=0.1,
                    length_includes_head=True, linewidth=2)
    ax.arrow(0, 0, v1[0], v1[1], color='crimson', **arrow_kw)
    ax.arrow(0, 0, v2[0], v2[1], color='darkorange', **arrow_kw)

    # labels on the arrows
    ax.text(v1[0] + 0.1, v1[1] + 0.1, f'v1 = {v1}', color='crimson',   fontsize=11)
    ax.text(v2[0] + 0.1, v2[1] - 0.2, f'v2 = {v2}', color='darkorange', fontsize=11)

    # center label showing area
    cx, cy = (v1 + v2) / 2
    ax.text(cx, cy, f'area = |det| = {area:.2f}',
            ha='center', va='center', fontsize=12,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

    ax.set_xlim(-0.5, max(v1[0], v2[0], v1[0]+v2[0]) + 1)
    ax.set_ylim(-0.5, max(v1[1], v2[1], v1[1]+v2[1]) + 1)
    ax.axhline(0, color='gray', linewidth=0.8)
    ax.axvline(0, color='gray', linewidth=0.8)
    ax.set_aspect('equal')
    ax.set_title(f'det = {det:.2f}   →   area = {area:.2f}', fontsize=13)
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

# --- Try these and observe what happens ---

# Two independent vectors: large area
plot_parallelogram(np.array([3.0, 0.0]), np.array([1.0, 2.0]))

# Nearly dependent: area shrinks
plot_parallelogram(np.array([3.0, 0.0]), np.array([3.0, 0.5]))

# Fully dependent (one is a multiple of the other): area = 0, det = 0, rank = 1
plot_parallelogram(np.array([3.0, 0.0]), np.array([1.5, 0.0]))

---

## Part 6: Practice Problems

Now it's your turn! For each problem, make a small change to the code. Look back at the examples if you need help.

---

### Problem 1: Rank-Nullity Check

Below are three matrices. **Your task:** for each one, compute the rank and the nullity using the functions we've used, then verify that rank + nullity = number of columns.

Fill in the three `___` blanks.

In [ ]:
# Problem 1

P1 = np.array([[1,2,3,4],[0,1,2,3],[0,0,1,2],[0,0,0,0]], dtype=float)
P2 = np.array([[1,0,1],[0,1,1],[1,1,2],[2,1,3]], dtype=float)
P3 = np.array([[2,4,6],[1,2,3],[3,6,9]], dtype=float)

for name, P in [("P1", P1), ("P2", P2), ("P3", P3)]:
    m, n = P.shape
    r = ___                         # fill in: np.linalg.matrix_rank(P)
    ns = linalg.null_space(P)
    nullity = ___                   # fill in: ns.shape[1]
    total = ___                     # fill in: r + nullity
    print(f"{name}: shape={P.shape}, rank={r}, nullity={nullity}, rank+nullity={total}, n={n}, check={total==n}")

---

### Problem 2: Is $\mathbf{b}$ in the Column Space?

Use the `is_in_column_space` function to test whether each vector $\mathbf{b}$ below is in the column space of matrix $C$.

**Before running the code**, look at matrix $C$ and the vectors. Can you predict the answer? (Hint: think about what rank(C) is.)

In [ ]:
# Problem 2

C = np.array([
    [1, 2],
    [3, 6],
    [2, 4]
], dtype=float)

print(f"rank(C) = {np.linalg.matrix_rank(C)}")
print(f"C has {C.shape[0]} rows and {C.shape[1]} columns.")
print()

b_test1 = np.array([2, 6, 4], dtype=float)   # a multiple of column 1
b_test2 = np.array([1, 2, 1], dtype=float)   # not a multiple of either column

print("Testing b_test1 = [2, 6, 4]:")
___    # fill in: is_in_column_space(C, b_test1, "b_test1")

print("Testing b_test2 = [1, 2, 1]:")
___    # fill in: is_in_column_space(C, b_test2, "b_test2")

---

### Problem 3: Balance the Iron Rust Reaction

Iron rusting is the reaction:

$$x_1 \, \text{Fe} + x_2 \, \text{O}_2 \longrightarrow x_3 \, \text{Fe}_2\text{O}_3$$

- Fe has 1 iron, 0 oxygen
- O₂ has 0 iron, 2 oxygen  
- Fe₂O₃ has 2 iron, 3 oxygen

**Your task:** Fill in the atom matrix below (reactants positive, products negative), then call `balance_equation` to find the coefficients.

In [ ]:
# Problem 3
# Columns: Fe,  O2,  Fe2O3
# Rows: Iron, Oxygen

#              Fe    O2    Fe2O3
A_rust = np.array([
    [ ___,   0,   ___  ],   # Iron:   1 in Fe, -2 in Fe2O3
    [ ___,   2,   ___  ],   # Oxygen: 0 in Fe, 2 in O2, -3 in Fe2O3
], dtype=float)

balance_equation(
    A_rust,
    ['Fe', 'O2', 'Fe2O3'],
    "Fe + O2 → Fe2O3"
)

---

### Problem 4: Balance Photosynthesis

Photosynthesis is the reaction by which plants make sugar from CO₂ and water:

$$x_1 \, \text{CO}_2 + x_2 \, \text{H}_2\text{O} \longrightarrow x_3 \, \text{C}_6\text{H}_{12}\text{O}_6 + x_4 \, \text{O}_2$$

Molecule composition:
- CO₂: C=1, H=0, O=2
- H₂O: C=0, H=2, O=1
- C₆H₁₂O₆ (glucose): C=6, H=12, O=6
- O₂: C=0, H=0, O=2

**Your task:** Build the complete atom matrix and call `balance_equation`. Remember: reactants get positive signs, products get negative signs.

In [ ]:
# Problem 4
# CO2 + H2O → C6H12O6 + O2
# Columns: CO2, H2O, C6H12O6, O2
# Rows: Carbon, Hydrogen, Oxygen

#                   CO2   H2O   C6H12O6   O2
A_photo = np.array([
    [ ___,  ___,   ___,    ___ ],   # Carbon
    [ ___,  ___,   ___,    ___ ],   # Hydrogen
    [ ___,  ___,   ___,    ___ ],   # Oxygen
], dtype=float)

balance_equation(
    A_photo,
    ['CO2', 'H2O', 'C6H12O6', 'O2'],
    "CO2 + H2O → C6H12O6 + O2"
)

---

### Problem 5: The Determinant and Linear Dependence

In the chemistry problems above, the atom matrix was never square — so we couldn't use the determinant directly. But when a matrix *is* square, the determinant gives us an instant rank test:

> **det(A) ≠ 0  ↔  rank = n  ↔  null space is just {0}**
> **det(A) = 0  ↔  rank < n  ↔  null space is nontrivial**

In other words: a zero determinant is a warning signal that your system has a "free direction" — some nonzero vector that the matrix sends to zero.

**(a)** Three 2×2 matrices are defined below. **Before running the code**, predict for each one:
- Is it full rank?
- Will the determinant be zero or nonzero?
- Does the null space contain anything besides the zero vector?

Write your predictions as comments (use #), then run the cell to check.

**(b)** Matrix `M3` has determinant zero. Find a nonzero vector **x** such that `M3 @ x` equals the zero vector. (Hint: look at the columns of `M3` — how are they related?)

**(c)** The `plot_parallelogram` function from earlier visualizes the determinant as an area. Call it on each matrix using its two columns as vectors. Do the plots match your predictions?

---

## Summary

Here's what you discovered in this lab:

| Concept | Definition | Chemistry meaning |
|---|---|---|
| **Rank** | # of independent rows/columns | # of independent atom-conservation constraints |
| **Null space** | All $\mathbf{x}$ with $A\mathbf{x}=\mathbf{0}$ | All valid coefficient vectors that balance the reaction |
| **Nullity** | Dimension of the null space | # of free parameters in the balanced equation |
| **Rank-Nullity Thm** | rank + nullity = $n$ | (constraints) + (free coefficients) = (# of molecules) |
| **Column space** | All possible outputs $A\mathbf{x}$ | The space of atom-count vectors the reaction can produce |
| **Determinant** | Area/volume scaling factor of a square matrix; zero iff rank < $n$ | N/A for atom matrices (not generally square), but signals when a square system has a nontrivial null space |

The null space is not just an abstract mathematical object.  It's the **solution set** to many real problems, from fields like chemistry, engineering, and data science.

## Submission Instructions:
After running the examples and answering the questions at the end of the lab, export your notebook as a pdf and upload that to Canvas for your assignment.